In [41]:
import pandas as pd
import numpy as np

# Load your data (adjust path/file type)
menu = pd.read_csv("menu_items.csv")
orders = pd.read_csv("order_details.csv")
resturants = pd.read_csv("restaurant_db_data_dictionary.csv")

In [42]:
menu.head()

,menu_item_id,item_name,category,price
0,101,Hamburger,American,12.95
1,102,Cheeseburger,American,13.95
2,103,Hot Dog,American,9.00
3,104,Veggie Burger,American,10.50
4,105,Mac & Cheese,American,7.00


In [43]:
orders.head()

,order_details_id,order_id,order_date,order_time,item_id
0,1,1,1/1/23,11:38:36 AM,109.0
1,2,2,1/1/23,11:57:40 AM,108.0
2,3,2,1/1/23,11:57:40 AM,124.0
3,4,2,1/1/23,11:57:40 AM,117.0
4,5,2,1/1/23,11:57:40 AM,129.0


In [44]:
resturants.head()

,Table,Field,Description
0,menu_items,menu_item_id,Unique ID of a menu item
1,menu_items,item_name,Name of a menu item
2,menu_items,category,Category or type of cuisine of the menu item
3,menu_items,price,Price of the menu item (US Dollars $)
4,order_details,order_details_id,Unique ID of an item in an order


In [45]:
print("=== MISSING VALUES ===")
print("\nMenu items:")
print(menu.isnull().sum())

print("\nOrders:")
print(orders.isnull().sum())

print("\nRestaurant data dictionary:")
print(resturants.isnull().sum())

=== MISSING VALUES ===

Menu items:
menu_item_id    0
item_name       0
category        0
price           0
dtype: int64

Orders:
order_details_id      0
order_id              0
order_date            0
order_time            0
item_id             137
dtype: int64

Restaurant data dictionary:
Table          0
Field          0
Description    0
dtype: int64


In [46]:
orders = orders.dropna()

In [47]:
print("\nOrders:")
print(orders.isnull().sum())


Orders:
order_details_id    0
order_id            0
order_date          0
order_time          0
item_id             0
dtype: int64


### Menu DataSet

In [48]:
menu['category'].unique()

array(['American', 'Asian', 'Mexican', 'Italian'], dtype=object)

In [49]:
menu['price'].agg(['min', 'max', 'mean'])

min      5.000000
max     19.950000
mean    13.285937
Name: price, dtype: float64

In [50]:
summary = (
    menu
    .groupby('category')
    .agg(
        avg_price=('price', 'mean'),
        total_items=('price', 'count')
    )
    .round(2)
    .sort_values(by='avg_price', ascending=False)
)

print(summary)

          avg_price  total_items
category                        
Italian       16.75            9
Asian         13.48            8
Mexican       11.80            9
American      10.07            6


In [51]:
idx = menu.groupby('category')['price'].idxmax()
most_expensive = menu.loc[idx, ['category', 'item_name', 'price']]

print(most_expensive)

    category         item_name  price
1   American      Cheeseburger  13.95
8      Asian  Korean Beef Bowl  17.95
29   Italian     Shrimp Scampi  19.95
17   Mexican     Steak Burrito  14.95


In [52]:
# Full pipeline
menu['price_category'] = np.select(
    [
        menu['price'] < 10,
        menu['price'].between(10, 15)
    ],
    ['Cheap', 'Mid-range'],
    default='Expensive'
)

analysis = menu.groupby(['category', 'price_category']).agg(
    avg_price=('price', 'mean'),
    total_items=('price', 'count')
)

print(analysis)

                         avg_price  total_items
category price_category                        
American Cheap            7.666667            3
         Mid-range       12.466667            3
Asian    Cheap            7.000000            2
         Expensive       17.466667            3
         Mid-range       13.800000            3
Italian  Expensive       17.392857            7
         Mid-range       14.500000            2
Mexican  Cheap            8.000000            2
         Mid-range       12.885714            7


In [53]:
# Merge (JOIN)
merged = pd.merge(
    orders,
    menu,
    left_on="item_id",
    right_on="menu_item_id",
    how="left"   # keeps all orders
)

# Optional: reorder columns (to match your format)
merged = merged[
    [
        "order_details_id",
        "order_id",
        "order_date",
        "order_time",
        "item_id",
        "item_name",
        "category",
        "price"
    ]
]

# Save to new CSV
merged.to_csv("combined_orders.csv", index=False)

print("✅ File saved as combined_orders.csv")

✅ File saved as combined_orders.csv


In [54]:
df.head()
print(df['item_id'].isna().sum())
# df.to_csv("combined_orders_cleaned.csv", index=False)

0


In [56]:
print(df.shape)

(12097, 8)


In [57]:
df["order_date"] = pd.to_datetime(df["order_date"], format="%m/%d/%y", errors="coerce")
df["order_time"] = pd.to_datetime(df["order_time"], format="%I:%M:%S %p", errors="coerce").dt.time

In [59]:
# Ensure correct types
df["item_id"]          = df["item_id"].astype(int)
df["price"]            = df["price"].astype(float)
df["order_id"]         = df["order_id"].astype(int)
df["order_details_id"] = df["order_details_id"].astype(int)
df["category"]         = df["category"].str.strip().str.title()
df["item_name"]        = df["item_name"].str.strip().str.title()
 
# Derived columns
df["month"]      = df["order_date"].dt.month
df["month_name"] = df["order_date"].dt.strftime("%b")
df["day_name"]   = df["order_date"].dt.day_name()
df["hour"]       = pd.to_datetime(df["order_time"].astype(str), format="%H:%M:%S", errors="coerce").dt.hour
df["week"]       = df["order_date"].dt.isocalendar().week.astype(int)
df["is_weekend"] = df["day_name"].isin(["Saturday", "Sunday"])
 
print("\n=== CLEANED SAMPLE ===")
print(df.head())


=== CLEANED SAMPLE ===
   order_details_id  order_id order_date order_time  item_id  \
0                 1         1 2023-01-01   11:38:36      109   
1                 2         2 2023-01-01   11:57:40      108   
2                 3         2 2023-01-01   11:57:40      124   
3                 4         2 2023-01-01   11:57:40      117   
4                 5         2 2023-01-01   11:57:40      129   

          item_name category  price  month month_name day_name  hour  week  \
0  Korean Beef Bowl    Asian  17.95      1        Jan   Sunday    11    52   
1     Tofu Pad Thai    Asian  14.50      1        Jan   Sunday    11    52   
2         Spaghetti  Italian  14.50      1        Jan   Sunday    11    52   
3   Chicken Burrito  Mexican  12.95      1        Jan   Sunday    11    52   
4  Mushroom Ravioli  Italian  15.50      1        Jan   Sunday    11    52   

   is_weekend  
0        True  
1        True  
2        True  
3        True  
4        True  


In [60]:
print("\n=== PRICE STATS ===")
print(df["price"].describe().round(2))
 
print("\n=== NUMPY STATS ===")
prices = df["price"].values
print(f"Mean:     ${np.mean(prices):.2f}")
print(f"Median:   ${np.median(prices):.2f}")
print(f"Std Dev:  ${np.std(prices):.2f}")
print(f"Variance: ${np.var(prices):.2f}")
print(f"Min:      ${np.min(prices):.2f}")
print(f"Max:      ${np.max(prices):.2f}")
print(f"Range:    ${np.ptp(prices):.2f}")
print(f"25th pct: ${np.percentile(prices, 25):.2f}")
print(f"75th pct: ${np.percentile(prices, 75):.2f}")


=== PRICE STATS ===
count    12097.00
mean        13.16
std          3.99
min          5.00
25%         10.50
50%         13.95
75%         16.50
max         19.95
Name: price, dtype: float64

=== NUMPY STATS ===
Mean:     $13.16
Median:   $13.95
Std Dev:  $3.99
Variance: $15.89
Min:      $5.00
Max:      $19.95
Range:    $14.95
25th pct: $10.50
75th pct: $16.50


In [61]:
# --- 4a. Revenue per order ---
order_revenue = (
    df.groupby("order_id")["price"]
    .sum()
    .reset_index()
    .rename(columns={"price": "order_total"})
)
print("\n=== ORDER REVENUE (top 5) ===")
print(order_revenue.sort_values("order_total", ascending=False).head())


=== ORDER REVENUE (top 5) ===
      order_id  order_total
435        440       192.15
2064      2075       191.05
1946      1957       190.10
325        330       189.70
2658      2675       185.10


In [62]:
# --- 4b. Category performance ---
cat_stats = df.groupby("category").agg(
    total_orders  = ("order_id", "count"),
    total_revenue = ("price", "sum"),
    avg_price     = ("price", "mean"),
    unique_items  = ("item_name", "nunique"),
).round(2).reset_index().sort_values("total_revenue", ascending=False)
print("\n=== CATEGORY STATS ===")
print(cat_stats)
 


=== CATEGORY STATS ===
   category  total_orders  total_revenue  avg_price  unique_items
2   Italian          2948       49462.70      16.78             9
1     Asian          3470       46720.65      13.46             8
3   Mexican          2945       34796.80      11.82             9
0  American          2734       28237.75      10.33             6


In [63]:
# --- 4c. Top 10 items ---
item_stats = df.groupby(["item_name", "category"]).agg(
    orders  = ("order_id", "count"),
    revenue = ("price", "sum"),
    price   = ("price", "mean"),
).round(2).reset_index().sort_values("orders", ascending=False)
print("\n=== TOP 10 ITEMS ===")
print(item_stats.head(10))
 


=== TOP 10 ITEMS ===
                item_name  category  orders   revenue  price
14              Hamburger  American     622   8054.90  12.95
10                Edamame     Asian     620   3100.00   5.00
16       Korean Beef Bowl     Asian     588  10554.60  17.95
3            Cheeseburger  American     583   8132.85  13.95
13           French Fries  American     571   3997.00   7.00
30          Tofu Pad Thai     Asian     562   8149.00  14.50
29            Steak Torta   Mexican     489   6821.55  13.95
26  Spaghetti & Meatballs   Italian     470   8436.50  17.95
17           Mac & Cheese  American     463   3241.00   7.00
9           Chips & Salsa   Mexican     461   3227.00   7.00


In [64]:
# --- 4d. Monthly revenue ---
monthly = df.groupby(["month", "month_name"])["price"].sum().reset_index()
monthly = monthly.sort_values("month").reset_index(drop=True)
monthly.columns = ["month", "month_name", "revenue"]
print("\n=== MONTHLY REVENUE ===")
print(monthly)


=== MONTHLY REVENUE ===
   month month_name   revenue
0      1        Jan  53816.95
1      2        Feb  50790.35
2      3        Mar  54610.60


In [65]:
# --- 4e. Day-of-week volume ---
day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
dow = df.groupby("day_name")["order_id"].count().reindex(day_order).reset_index()
dow.columns = ["day", "orders"]
print("\n=== DAY OF WEEK ===")
print(dow)


=== DAY OF WEEK ===
         day  orders
0     Monday    1988
1    Tuesday    1760
2  Wednesday    1522
3   Thursday    1667
4     Friday    1802
5   Saturday    1599
6     Sunday    1759


In [66]:
# --- 4f. Hourly distribution ---
hourly = df.groupby("hour")["order_id"].count().reset_index()
hourly.columns = ["hour", "orders"]

In [67]:
# --- 4g. Weekend vs weekday ---
wk_compare = df.groupby("is_weekend")["price"].agg(["sum","count","mean"]).round(2)
wk_compare.index = ["Weekday", "Weekend"]
wk_compare.columns = ["revenue", "orders", "avg_price"]
print("\n=== WEEKEND vs WEEKDAY ===")
print(wk_compare)
 


=== WEEKEND vs WEEKDAY ===
           revenue  orders  avg_price
Weekday  114820.55    8739      13.14
Weekend   44397.35    3358      13.22


In [68]:
# --- 4h. Price segmentation with numpy ---
bins   = [0, 10, 15, 20, np.inf]
labels = ["Budget (<$10)", "Mid ($10–15)", "Premium ($15–20)", "Luxury ($20+)"]
df["price_tier"] = pd.cut(df["price"], bins=bins, labels=labels)
tier_counts = df["price_tier"].value_counts().sort_index()
print("\n=== PRICE TIERS ===")
print(tier_counts)


=== PRICE TIERS ===
price_tier
Budget (<$10)       2814
Mid ($10–15)        5547
Premium ($15–20)    3736
Luxury ($20+)          0
Name: count, dtype: int64


In [69]:
# --- 4i. Pivot: category × month revenue ---
pivot = df.pivot_table(
    values="price", index="category",
    columns="month_name", aggfunc="sum"
).fillna(0).round(2)
print("\n=== PIVOT: CATEGORY × MONTH ===")
print(pivot)
 


=== PIVOT: CATEGORY × MONTH ===
month_name       Feb       Jan       Mar
category                                
American     8947.15   9284.85  10005.75
Asian       15075.70  15588.50  16056.45
Italian     15545.05  16727.75  17189.90
Mexican     11222.45  12215.85  11358.50


In [70]:
# --- 4j. Top item per category ---
top_per_cat = (
    item_stats.groupby("category")
    .apply(lambda x: x.nlargest(1, "orders"))
    .reset_index(drop=True)[["category","item_name","orders","revenue"]]
)
print("\n=== TOP ITEM PER CATEGORY ===")
print(top_per_cat)
 
# --- 4k. Slow movers (bottom 5 items) ---
slow = item_stats.nsmallest(5, "orders")[["item_name","category","orders","revenue"]]
print("\n=== SLOW MOVERS (bottom 5) ===")
print(slow)
 


=== TOP ITEM PER CATEGORY ===
   category              item_name  orders  revenue
0  American              Hamburger     622  8054.90
1     Asian                Edamame     620  3100.00
2   Italian  Spaghetti & Meatballs     470  8436.50
3   Mexican            Steak Torta     489  6821.55

=== SLOW MOVERS (bottom 5) ===
             item_name category  orders  revenue
6        Chicken Tacos  Mexican     123  1469.85
22         Potstickers    Asian     205  1845.00
1       Cheese Lasagna  Italian     207  3208.50
28         Steak Tacos  Mexican     214  2985.30
2   Cheese Quesadillas  Mexican     233  2446.50


/var/folders/wr/mm1996jd4p9grx5zzl61s8h40000gn/T/ipykernel_32337/1723524703.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.nlargest(1, "orders"))


In [76]:
df.head()

,order_details_id,order_id,order_date,order_time,item_id,item_name,category,price,month,month_name,day_name,hour,week,is_weekend,price_tier
0,1,1,2023-01-01,11:38:36,109,Korean Beef Bowl,Asian,17.95,1,Jan,Sunday,11,52,True,Premium ($15–20)
1,2,2,2023-01-01,11:57:40,108,Tofu Pad Thai,Asian,14.50,1,Jan,Sunday,11,52,True,Mid ($10–15)
2,3,2,2023-01-01,11:57:40,124,Spaghetti,Italian,14.50,1,Jan,Sunday,11,52,True,Mid ($10–15)
3,4,2,2023-01-01,11:57:40,117,Chicken Burrito,Mexican,12.95,1,Jan,Sunday,11,52,True,Mid ($10–15)
4,5,2,2023-01-01,11:57:40,129,Mushroom Ravioli,Italian,15.50,1,Jan,Sunday,11,52,True,Premium ($15–20)


In [78]:
# Replace 'combined_orders.csv' with your actual filename if it's different
df.to_csv('combined_orders.csv', index=False)
